# Beamforming Microphone Array Data

In this task you will be given a real recording from one of Squarehead's microphone arrays. A microphone array is a collection of many microphones organized in a given pattern to be able to sample the spatial properties of a propagating wave field. The Squarehead microphone array is also equipped with a video camera that makes it possible to overlay a sound image on top of the video image. With this, it is possible to see the position of sound sources, and by that aim/steer the array response/beam in the direction of the sound sources.

The recording you are given is of two persons speaking together, and the task is to attenuate speaker 2 as much as possible while aiming the array (digitally) at speaker 1. 

## The data
You are given a folder containing the array data and some metadata. Inside this folder you will find:
* An `.int16` file containing the array audio data with 16 bit (or 2 bytes) per sample
* A metadata json file
* A folder containing all the video frames from the recording.

### Audio
The int16 file is saved in time-major-order, meaning that the first time sample for all microphones comes before the second time sample for all microphones. If $s_{n,m}$ is the n'th time sample from the m'th microphone, $N$ is the total number of time samples per microphone, and $M$ is the total number of microphones, the data is saved as:

$[s_{0,0}, s_{0,1}, ... , s_{0,M-1}, ... , s_{n,m}, ... , s_{N-1,0}, ... , s_{N-1,M-2}, s_{N-1,M-1}]$.

### Metadata
The metadata files has the following structure:
```
{
    "sampleRateInHz":double,
    "startTimestamp":str,
    "temperatureInC":double,
    "targetFocalPoints":list(dict),
    "arrayLookDirection":list
    "arrayCoordinatesInMeters":{
        "x":list,
        "y":list,
        "z":list
    }
}
```

As you see, here you will find all the data needed to perform a beamforming operation. The `"targetFocalPoints"` is a list of dict objects with the following keys:
```
{
    "label":str,
    "point":[x,y,z],
    "intervals":[[t_start, t_end], ..., [t_start, t_end]]
}
```
We define the first label in this list as speaker 1, and the second as speaker 2. The `"intervals"` key contains intervals (in seconds) for when the given source is talking (approximately).

The `"arrayLookDirection"` holds the direction of the array relative to the array coordinates, and it should be `[0, 0, 1]`.

### Video
There is a folder named `video` that contains all the video frames from the recording as JPEG images. The images are named with a Epoch nano second timestamp (hence, number of nanoseconds since January 1, 1970). The images are just provide for completeness as somebody might find it interesting to go the whole way of beamforming audio images and superimpose those images on top of the video images. Maybe somebody are able to do it for the whole file and create a movie? Note that no audio-plane-vs-image-plane calibration is provided, so it will not be possible to match the two images perfectly.

## Imports and constants

In [1]:
# Imports
import json
from pathlib import Path
import numpy as np
from IPython.display import Audio
import os, sys
import plotly.express as px

In [ ]:
# Constants
INPUT_PATH = Path(r'C:\Users\theod\Documents\uio\in5450\array_signal_processing_in5450\data\in5450_oblig2_extra_task_data\in5450_oblig2_extra_task_data')
INT16_FILE = INPUT_PATH / 'data.int16' / 'data.int16'
METADATA_FILE = INPUT_PATH / 'metadata.json'
VIDEO_FOLDER = INPUT_PATH / 'video'

assert(INT16_FILE.exists())
assert(METADATA_FILE.exists())

In [3]:
#! unzip $INT16_FILE # uncomment this and run if you want to unzip the int16 data in a colab session

## Helper functions

In [4]:
def load_json(filename):
    with open(filename, 'r') as f:
        return json.load(f)

In [5]:
def channel_data_block_generator(file_path, n_channels, n_time_samples_per_block):
    with open(file_path, 'rb') as file:
        block_size = n_channels*n_time_samples_per_block*2
        while True:
            chunk = file.read(block_size)
            if not chunk or len(chunk) != block_size:
                break
            yield np.frombuffer(chunk, dtype=np.int16).reshape((n_time_samples_per_block, n_channels)).T

In [6]:
def read_single_block(file_path, n_channels, n_time_samples_per_block):
    for block in channel_data_block_generator(file_path, n_channels, n_time_samples_per_block):
        return block
    raise Exception('No block found')

In [7]:
def block_with_padding(file_path, n_channels, n_time_samples_per_block, sample_padding):
    fifo = []
    for block in channel_data_block_generator(file_path, n_channels, n_time_samples_per_block):
        fifo.append(block)
        if len(fifo) > 3:
            fifo.pop(0)
        if len(fifo) == 3:
            yield convert_block_to_float(np.concatenate((fifo[0][:,-sample_padding:], fifo[1], fifo[2][:,0:sample_padding]), axis=1))

In [8]:
def single_channel_data(file_path, fs, channel, n_channels):
    block_size = 256
    channel_data = [convert_block_to_float(block[channel,:]) for block in channel_data_block_generator(file_path, n_channels, block_size)]
    return np.concatenate(tuple(channel_data))

In [9]:
def convert_block_to_float(block):
    return np.cast[float](block) / float(np.iinfo(np.int16).max)

In [10]:
def db_rms(audio):
    return 10*np.log10(np.mean(audio*audio))

In [11]:
def snr(audio, fs, signal_interval, noise_interval):
    s = audio[int(signal_interval[0]*fs):int(signal_interval[1]*fs)]
    n = audio[int(noise_interval[0]*fs):int(noise_interval[1]*fs)]
    return db_rms(s) - db_rms(n)

In [12]:
def get_silent_interval(metadata):
    end = sys.float_info.max
    for p in metadata['targetFocalPoints']:
        for i in p['intervals']:
            end = min(end, i[0])
    padding = 0.5
    return [0, max(0, end - padding)]

In [13]:
def clip_audio(audio, fs, interval):
    return audio[int(fs*interval[0]):int(fs*interval[1])]   

## Investigating the audio data

Let us load some of the audio data and play it.

First we need to load the metadata.

In [14]:
metadata = load_json(METADATA_FILE)
Xm = np.array(metadata['arrayCoordinatesInMeters']['x'])
Ym = np.array(metadata['arrayCoordinatesInMeters']['y'])
Zm = np.array(metadata['arrayCoordinatesInMeters']['z'])
fs = metadata['sampleRateInHz']
M = len(Xm) # number of microphone elements
assert(M == 60) # the array has 60 microphones

Let us plot the array to get an idea of the array geometry.

In [15]:
fig = px.scatter(x=Xm, y=Ym, hover_name=[f"Index {i}" for i in range(M)]) # the Zm vector is just zeros
fig.update_traces(marker=dict(size=40))
fig.update_yaxes(
    scaleanchor = "x",
    scaleratio = 1,
)
fig.show()

So we see that this is a 2D uniform linear array. Try to add `color=range(M)` to the scatter call and see if you see something strange in the center of the array. Stay calm, it should be like this. There is nothing wrong.

Now read a large block of channel data and play one of the channels.

In [16]:
length_in_seconds = metadata['targetFocalPoints'][0]['intervals'][0][1]
block = convert_block_to_float(read_single_block(INT16_FILE, M, int(length_in_seconds*fs)))
print(block.shape)

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\theod\\Documents\\uio\\in5450\\array_signal_processing_in5450\\data\\in5450_oblig2_extra_task_data\\in5450_oblig2_extra_task_data\\data.int16'

In [ ]:
Audio(block[0,:], rate=fs, autoplay=True)

## Beamforming

### The simple beam

The simplest way of forming a beam is to just sum all the channels. In other words no steering, resulting in a beam "steered" straight ahead with uniform weights.

In [ ]:
beam = np.sum(block, 0) / float(M)

Now let us play this beam.

In [ ]:
Audio(beam, rate=fs, autoplay=True)

When comparing with the single channel audio we hear the effect of summing multiple elements on the background noise level (white noise) and also for sources that do not happen to be positioned straight in front of the array.

We can check this by measuring the signal-to-noise ratio (SNR) between a segment in the audio where we know there is signal and a section where we know it is just noise. 

In [ ]:
signal_interval = metadata['targetFocalPoints'][0]['intervals'][0]
noise_interval = get_silent_interval(metadata)
print(signal_interval)
print(noise_interval)

single_channel_snr = snr(block[0,:], fs, signal_interval, noise_interval)
print(f"Single channel SNR: {single_channel_snr} dB")
beam_snr = snr(beam, fs, signal_interval, noise_interval)
print(f"Beam SNR: {beam_snr} dB")

The bemforming filter operation was able to attenuate the single-channel noise by around 5.4 dB.

### A dynamic steerable beam 

A static forward-facing beam is boring. We need to make it dynamically steerable to make it more interesting.

The first thing we need is to calculate the time delay for all the elements in our array when given a focal point. Since we are working with discrete samples at a fixed sample rate, we also need to convert our time delays into discrete index delays. Also note how the delay calculation depends on the sound speed in air (`c`).

In [ ]:
def calc_index_delays(Xm, Ym, Zm, focal_point, fs, c=340):
    x, y, z = focal_point
    time_delays = np.squeeze(np.sqrt((Xm-x)**2 + (Ym-y)**2 + (Zm-z)**2) - np.linalg.norm(focal_point)) / c
    return np.round(fs * time_delays)

To check that this is somewhat correct we calculate the delays for a source located at the first element in the first line of the array.

In [ ]:
index_delays = calc_index_delays(Xm, Ym, Zm, [Xm[0], Ym[0], Zm[0]], fs)
print(index_delays[0:10])

Since the center of the array is in `[0,0,0]`, we get a zero (or close to zero) index shift at the center, negative shifts for the elements closer to the focal point, and positive shifts for the elements further away from the focal point than the array center. This fits nicely with the fact that an impulse signal originating from the specified `focal_point` will be sampled first by the microphone closest to the focal point, and hence this data will be located earlier in memory (or at negative shifts) compared to microphones further away.

We can make a plot of the array with the index shifts color encoded to make it more visual.

In [ ]:
fig = px.scatter(x=Xm, y=Ym, hover_name=[f"Index shift {s}" for s in index_delays], color=index_delays)
fig.update_traces(marker=dict(size=40))
fig.update_yaxes(
    scaleanchor = "x",
    scaleratio = 1,
)
fig.show()

Try to set `focal_point=[0,0,0]` and redo the plot.

To steer the beam in a given direction we need to apply delays to our channel data before summing. Let us assemble the pieces and write a beamforming function. Since some delays can be negative we need to use a padded buffer and make sure that the padding is at least larger than the largest index delay.

In [ ]:
def beamform(file_path, fs, Xm, Ym, Zm, focal_point, Wm=None, c=340.0):
    M = len(Xm)
    Wm = (np.ones(M) / float(M)) if Wm is None else Wm
    if M != len(Wm):
        raise Exception(f'len(Wm) != len(Xm)')
    index_delays = calc_index_delays(Xm, Ym, Zm, focal_point, fs, c)
    max_delay = np.max(np.abs(index_delays))
    block_size = 256
    padding = int(max_delay)
    beam = []

    for block in block_with_padding(file_path, M, block_size, padding):
        sum = np.zeros(block_size)
        for m in range(M):
            index = padding + int(index_delays[m])
            sum += Wm[m] * block[m, index:index+block_size]
        beam.append(sum)

    return np.concatenate(tuple(beam))
    

Now let us try to steer towards speaker 1 and the speaker 2 and play the result.

In [ ]:
focal_point_1 = metadata['targetFocalPoints'][0]['point']
print(focal_point_1)
beam_1 = beamform(INT16_FILE, fs, Xm, Ym, Zm, focal_point_1)

In [ ]:
Audio(beam_1, rate=fs, autoplay=True)

In [ ]:
focal_point_2 = metadata['targetFocalPoints'][1]['point']
print(focal_point_2)
beam_2 = beamform(INT16_FILE, fs, Xm, Ym, Zm, focal_point_2)

In [ ]:
Audio(beam_2, rate=fs, autoplay=True)

Let us also load a single channel of data for the whole file for comparison.

In [ ]:
single_channel = single_channel_data(INT16_FILE, fs, 0, M)
Audio(single_channel, rate=fs, autoplay=True)

## Measuring the signal-to-noise ratio (SNR)

Now let us measure the increase in SNR that we were able to achieve using beamforming. Again we do this by measuring the dB RMS level for a silent interval in the file and compare it with an interval where one of the persons is talking. If we do the measurements for an interval where person 2 is talking, but we aim the array at person 1, we will get a measure of how well we are able to attenuate person 2 (also when person two is talking).

In [ ]:
interval_person_1 = metadata['targetFocalPoints'][0]['intervals'][0]
print(interval_person_1)
noise_interval = get_silent_interval(metadata)
print(noise_interval)

# single channel SNR
single_channel_snr = snr(single_channel, fs, interval_person_1, noise_interval)
print(f"Single channel SNR: {single_channel_snr} dB")

# beam SNR when steering at person 2 and person 1 is talking 
print(focal_point_2)
beam_2 = beamform(INT16_FILE, fs, Xm, Ym, Zm, focal_point_2)
beam_snr = snr(beam_2, fs, interval_person_1, noise_interval)
print(f'Off-beam SNR: {beam_snr} dB')

# beam SNR when steering at person 1 and person 1 is talking 
print(focal_point_1)
beam_1 = beamform(INT16_FILE, fs, Xm, Ym, Zm, focal_point_1)
beam_snr = snr(beam_1, fs, interval_person_1, noise_interval)
print(f'Beam SNR: {beam_snr} dB')


These numbers are not that impressive. We where just able to attenuate speaker 2 with just a fraction of a dB. However, this broadband SNR value does not tell the full story, and we will see why in the next section.

### Sub-band SNR
Since the directivity of the array increases with frequency (and is quite omni directional for lower frequencies) we will not see that much of an improvement when using a broadband dB level measure. To better see the effect of the array we can try to plot the SNR improvements as a function of frequency.

In [ ]:
from scipy.signal import welch
f, psd_1 = welch(clip_audio(beam_1, fs, interval_person_1), fs)
f, psd_2 = welch(clip_audio(beam_2, fs, interval_person_1), fs)
f, psd_single = welch(clip_audio(single_channel, fs, interval_person_1), fs)

psd_1_db = 10*np.log10(psd_1)
psd_2_db = 10*np.log10(psd_2)
psd_single_db = 10*np.log10(psd_single)

import pandas as pd
df = pd.DataFrame({
    "Signal frequencies [Hz]":f, 
    "On target":psd_1_db, 
    "Off target":psd_2_db,
    "Single channel":psd_single_db 
})
fig = px.line(
    df, 
    x="Signal frequencies [Hz]", 
    y=["On target", "Off target", "Single channel"],
    hover_name=[f"SNR: {db1 - db2}" for db1, db2 in zip(psd_1_db, psd_2_db)]
)
fig.update_layout(yaxis_title="Power [dB]")
fig.show()

Now we see that the array provides multiple dBs of attenuation from 1.5 kHz and above. From 15 kHz and above we do not have the same increase, but this is due to the fact that the voices in the given recording does not contain that much energy for these frequencies.

## Extra task 1
How much are you able to attenuate speaker 2 just by using array weights or other techniques? In other words, are you able to increase the SNR values measured above.


In [ ]:
# Insert code here. Feel free to add as many code and text blocks as you like.

## Extra task 2
Use the beamforming function to generate an audio image of the scene. You do this by calculating a grid of focal points (aka audio pixels) instead of a single point.

One pixel in an acoustic image is typically generated by summing the signal power for a given interval in time. This interval could be the whole length of the file or a smaller interval. An interval of 100 ms will give audio images at a rate of 10 Hz. Note how you can use `block_with_padding(...)` to get a block of a given length and padding as done inside the `beamforming` function. Feel free to change the `beamforming` function into a generator, or create a new generator that returns the `rms` of a set of beam block segment (one segment for each focal point in the grid) one at the time.

In [ ]:
# Insert code here. Feel free to add as many code and text blocks as you like.